# EEE 457 Transmissão de Energia Elétrica
## Escola Politécnica
## Universidade Federal do Rio de Janeiro
### Antonio C. S. Lima
Outubro 2025
### Cálculo de parâmetros unitários em linhas de transmissão aéreas 

In [1]:
# carrega as bibliotecas
from __future__ import annotations

import numpy as np
from numpy import pi, sqrt, log, exp, diag, eye, kron
from numpy.linalg import inv, eig
from scipy.special import iv, kv
import matplotlib.pyplot as plt
import pandas as pd
from typing import Iterable, Tuple, Optional, Union
import warnings 
import os

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "sans-serif",
    "font.size": 14,
    "font.sans-serif": "Helvetica",
})

In [2]:
# constantes importantes
µ0 = 4e-7 * pi  # permeabilidade do vácuo (H/m)
ε0 = 8.854187817e-12  # permissividade do vácuo (F/m)
c0 = 1 / sqrt(µ0 * ε0)  # velocidade da luz no vácuo (m/s)
η0 = sqrt(µ0 / ε0)  # impedância característica do vácuo (Ohms)

# funções auxiliares
class TransmissionLineError(Exception):
    """Custom exception for transmission line parameter errors."""
    pass

def _validate_positive(value: float, name: str) -> None:
    """Validate that a value is positive."""
    if value <= 0:
        raise ValueError(f"{name} must be positive, got {value}")

def _validate_non_negative(value: float, name: str) -> None:
    """Validate that a value is non-negative."""
    if value < 0:
        raise ValueError(f"{name} must be non-negative, got {value}")

def zint_tubo(
    omega: complex, 
    rhoc: float, 
    rf: float, 
    rint: float, 
    mur: float = 1.0, 
    mu: float = µ0
) -> complex:
    """
    Internal impedance of tubular conductor.
    
    Parameters:
    -----------
    omega : complex
        Angular frequency [rad/s]
    rhoc : float
        Effective resistivity [Ω·m]
    rf : float
        Outer radius [m]
    rint : float
        Inner radius [m] (0 for solid conductor)
    mur : float
        Relative permeability
    mu : float
        Magnetic permeability [H/m]
    
    Returns:
    --------
    complex
        Internal impedance per unit length [Ω/m]
    """
    _validate_positive(rhoc, "rhoc")
    _validate_positive(rf, "rf")
    _validate_non_negative(rint, "rint")
    
    if rint >= rf:
        raise ValueError("Inner radius must be less than outer radius")
    
    eta_c = sqrt(1j * omega * mur * mu / rhoc)
    ri = max(rint, 1e-12)  # Avoid numerical issues
    
    # Handle very small arguments for Bessel functions
    if abs(eta_c * ri) < 1e-8 or abs(eta_c * rf) < 1e-8:
        warnings.warn("Small argument approximation used for Bessel functions")
        return (rhoc / (pi * (rf**2 - ri**2))) * (1 + 1j * omega * mur * mu * (rf**2 + ri**2) / (4 * rhoc))
    
    den = kv(1, eta_c * ri) * iv(1, eta_c * rf) - kv(1, eta_c * rf) * iv(1, eta_c * ri)
    num = kv(1, eta_c * ri) * iv(0, eta_c * rf) + kv(0, eta_c * rf) * iv(1, eta_c * ri)
    
    return (rhoc * eta_c) * num / (2 * pi * rf * den)

# usado para os cabos para-raios que são sólidos
def zin(
    omega: complex, 
    rhopr: float, 
    rpr: float, 
    mur: float = 90.0, 
    mu: float = µ0
) -> complex:
    """
    Internal impedance of solid cylindrical conductor.
    
    Parameters:
    -----------
    omega : complex
        Angular frequency [rad/s]
    rhopr : float
        Effective resistivity [Ω·m]
    rpr : float
        Radius [m]
    mur : float
        Relative permeability
    mu : float
        Magnetic permeability [H/m]
    
    Returns:
    --------
    complex
        Internal impedance per unit length [Ω/m]
    """
    _validate_positive(rhopr, "rhopr")
    _validate_positive(rpr, "rpr")
    
    etapr = sqrt(1j * omega * mu * mur / rhopr)
    
    # Small argument approximation
    if abs(etapr * rpr) < 1e-8:
        return (rhopr / (pi * rpr**2)) * (1 + 1j * omega * mur * mu * rpr**2 / (4 * rhopr))
    
    return (etapr * rhopr) * iv(0, etapr * rpr) / (2 * pi * rpr * iv(1, etapr * rpr))

def _coaxial_impedance_element(
    omega: complex, 
    r_outer: float, 
    r_inner: float, 
    rho: Optional[float] = None,
    mur: float = 1.0, 
    mu: float = µ0
) -> complex:
    """Helper function for coaxial impedance elements."""
    if rho is None:
        # Dielectric impedance
        return 1j * omega * mur * mu / (2 * pi) * log(r_outer / r_inner)
    else:
        # Sheath impedance
        eta = sqrt(1j * omega * mur * mu / rho)
        den = iv(1, eta * r_outer) * kv(1, eta * r_inner) - iv(1, eta * r_inner) * kv(1, eta * r_outer)
        return (rho * eta) / (2 * pi * r_inner * den) * (
            iv(0, eta * r_inner) * kv(1, eta * r_outer) + kv(0, eta * r_inner) * iv(1, eta * r_outer)
        )

def z_solo(
    omega: complex, 
    r: float, 
    h1: float, 
    h2: float, 
    sigma_solo: complex, 
    mu: float = µ0
) -> complex:
    """
    Carson-type complex ground return impedance.
    
    Parameters:
    -----------
    omega : complex
        Angular frequency [rad/s]
    r : float
        Horizontal separation [m]
    h1 : float
        Height of conductor 1 [m]
    h2 : float
        Height of conductor 2 [m]
    sigma_solo : complex
        Soil conductivity [S/m]
    mu : float
        Magnetic permeability [H/m]
    
    Returns:
    --------
    complex
        Ground return impedance [Ω/m]
    """
    _validate_non_negative(r, "r")
    _validate_positive(h1, "h1")
    _validate_positive(h2, "h2")
    
    eta_solo = sqrt(1j * omega * mu * sigma_solo)
    a = h1 + h2
    b = h1 - h2
    r1 = sqrt(r**2 + b**2)
    r2 = sqrt(r**2 + a**2)
    
    term1 = kv(0, eta_solo * r1)
    term2 = kv(2, eta_solo * r2)
    term3 = 2.0 * exp(-a * eta_solo) * (1.0 + a * eta_solo) / (eta_solo**2 * r2**2)
    
    return 1j * omega * mu / (2 * pi) * (term1 + ((a**2 - r**2) / r2**2) * (term2 - term3))


In [3]:
# manipulação das matrizes de Z, Y

# para eliminação dos cabos para-raios
def kron_reduction(
    matrix: np.ndarray, 
    nc: int, 
    npr: int
) -> np.ndarray:
    """
    Kron reduction for impedance matrices.
    
    Parameters:
    -----------
    matrix : np.ndarray
        Full matrix to reduce
    nc : int
        Total number of conductors
    npr : int
        Number of ground wires to eliminate
    
    Returns:
    --------
    np.ndarray
        Reduced matrix
    """
    if npr == 0:
        return matrix
    
    if nc <= npr:
        raise ValueError("Number of ground wires must be less than total conductors")
    
    y_full = inv(matrix)
    return y_full[:nc - npr, :nc - npr]

# para reduçao de feixes para calcular condutor equivalente 
def bundle_reduction(
    matrix: np.ndarray, 
    nb: int, 
    nf: int
) -> np.ndarray:
    """
    Reduce bundled conductors to equivalent single conductors.
    
    Parameters:
    -----------
    matrix : np.ndarray
        Full admittance matrix
    nb : int
        Number of subconductors per bundle
    nf : int
        Number of final equivalent conductors
    
    Returns:
    --------
    np.ndarray
        Reduced matrix
    """
    if matrix.shape[0] != nb * nf or matrix.shape[1] != nb * nf:
        raise ValueError("Matrix dimensions must match nb * nf")
    
    reduced = np.zeros((nf, nf), dtype=np.complex128)
    
    for m in range(nf):
        for n in range(nf):
            i0, i1 = nb * m, nb * (m + 1)
            j0, j1 = nb * n, nb * (n + 1)
            reduced[m, n] = np.sum(matrix[i0:i1, j0:j1])
    
    return reduced


In [4]:
# funcao que calcula as matrizes por unidade de comprimento
def cZYlt(
    omega: complex,
    x: Iterable[float],
    y: Iterable[float],
    sigmas: complex,
    rdc: float,
    rf: float,
    rint: float,
    npr: int = 0,
    rdcpr: Optional[float] = None,
    rpr: Optional[float] = None,
    nb: int = 1,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute per-unit-length (Z, Y) for overhead transmission lines.
    
    Parameters:
    -----------
    omega : complex
        Angular frequency [rad/s]
    x, y : Iterable[float]
        Conductor coordinates [m]
    sigmas : complex
        Soil conductivity [S/m]
    rdc : float
        DC resistance of phase conductor [Ω/m]
    rf : float
        Outer radius of phase conductor [m]
    rint : float
        Inner radius of phase conductor [m] (0 for solid)
    npr : int
        Number of ground wires
    rdcpr : float, optional
        DC resistance of ground wire [Ω/m]
    rpr : float, optional
        Radius of ground wire [m]
    nb : int
        Bundle size (subconductors per phase)
    
    Returns:
    --------
    Tuple[np.ndarray, np.ndarray]
        Impedance and admittance matrices [Ω/m, S/m]
    """
    # Input validation
    x_arr = np.asarray(x, dtype=float)
    y_arr = np.asarray(y, dtype=float)
    
    if x_arr.shape != y_arr.shape:
        raise ValueError("x and y must have the same shape")
    
    nc = x_arr.size
    phases = nc - npr
    
    if phases <= 0:
        raise ValueError("Number of phases must be positive")
    
    if npr > 0 and (rdcpr is None or rpr is None):
        raise ValueError("rdcpr and rpr must be provided when npr > 0")
    
    # Calculate effective resistivities
    rhoc = rdc * pi * (rf**2 - max(rint, 0)**2)
    
    if npr > 0:
        rhopr = rdcpr * pi * rpr**2
    
    # Internal impedance matrix
    z_phase = zint_tubo(omega, rhoc, rf, rint) if rint > 0 else zin(omega, rhoc, rf, 1.0)
    
    z_diag = [z_phase] * phases
    if npr > 0:
        z_gw = zin(omega, rhopr, rpr, mur=90.0)
        z_diag.extend([z_gw] * npr)
    
    Z_int = np.diag(z_diag).astype(np.complex128)
    
    # External impedance matrix (vectorized)
    p = sqrt(1.0 / (1j * omega * µ0 * sigmas))
    x_diff = x_arr[:, None] - x_arr[None, :]
    y_sum = y_arr[:, None] + y_arr[None, :]
    y_diff = y_arr[:, None] - y_arr[None, :]
    
    radii = np.full(nc, rf)
    if npr > 0:
        radii[phases:] = rpr
    
    # Off-diagonal elements
    off_diag_mask = ~np.eye(nc, dtype=bool)
    numerator = x_diff**2 + (2 * p + y_sum)**2
    denominator = x_diff**2 + y_diff**2
    Z_ext_off = 1j * omega * µ0 / (2 * pi) * 0.5 * log(numerator / denominator)
    
    # Diagonal elements
    Z_ext_diag = 1j * omega * µ0 / (2 * pi) * log(2.0 * (y_arr + p) / radii)
    
    Z_ext = np.where(off_diag_mask, Z_ext_off, np.diag(Z_ext_diag))
    
    # Full impedance matrix
    Z_full = Z_int + Z_ext
    
    # Reduction
    if npr > 0:
        Z_reduced = inv(kron_reduction(Z_full, nc, npr))
        if nb > 1:
            nf = phases // nb
            Z_reduced = inv(bundle_reduction(Z_reduced, nb, nf))
    else:
        Z_reduced = Z_full
    
    # Potential coefficient matrix
    num = x_diff**2 + y_sum**2
    den = x_diff**2 + y_diff**2
    P = 0.5 * log(num / den)
    np.fill_diagonal(P, log(2.0 * y_arr / radii))
    
    # Admittance matrix
    if npr > 0:
        P_reduced = kron_reduction(P, nc, npr)
        if nb > 1:
            nf = phases // nb
            P_reduced = bundle_reduction(P_reduced, nb, nf)
        Y_reduced = 1j * omega * 2 * pi * ε0 * P_reduced
    else:
        Y_reduced = 1j * omega * 2 * pi * ε0 * inv(P)
    
    # Add small conductance for numerical stability
    Y_reduced += 1e-12 * np.eye(Y_reduced.shape[0], dtype=np.complex128)
    
    return Z_reduced, Y_reduced

# matriz de admitância nodal para o circuito da linha de transmissão
def ynLT(
    Z: np.ndarray, 
    Y: np.ndarray, 
    length: float
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute nodal admittance matrices for a transmission line.
    
    Parameters:
    -----------
    Z : np.ndarray
        Series impedance matrix [Ω/m]
    Y : np.ndarray
        Shunt admittance matrix [S/m]
    length : float
        Line length [m]
    
    Returns:
    --------
    Tuple[np.ndarray, np.ndarray]
        Y11 and Y12 nodal admittance matrices
    """
    _validate_positive(length, "length")
    
    Z_total = Z * length
    Y_total = Y * length
    
    # Eigenvalue decomposition
    gamma_squared = Z_total @ Y_total
    eigenvalues, eigenvectors = eig(gamma_squared)
    gamma = sqrt(eigenvalues)
    
    # Hyperbolic functions
    exp_neg_gamma_l = exp(-gamma * length)
    coth_gamma_l = (1 + exp_neg_gamma_l**2) / (1 - exp_neg_gamma_l**2)
    csch_gamma_l = 2 * exp_neg_gamma_l / (1 - exp_neg_gamma_l**2)
    
    # Transform to original basis
    T = eigenvectors
    T_inv = inv(T)
    
    Y11 = inv(Z_total) @ T @ diag(gamma * coth_gamma_l) @ T_inv
    Y12 = -inv(Z_total) @ T @ diag(gamma * csch_gamma_l) @ T_inv
    
    return Y11, Y12

## Alguns casos testes


## Primeira configuração: 
#### Circuito 345 kV (345CSH1) com dois condutores Tern

In [18]:
# dados do circuito
# comprimento do feixe
s = 0.4572  # m
f = 19.1  # m flecha dos condutores
fpr = 15.24  # m flecha dos para-raios
xc = np.array([-8.5 - s/2,  -8.5 + s/2, - s/2, s/2, 8.5 - s/2, 8.5 + s/2, -6.25, 6.25] ) # m
yc = np.array([28.4, 28.4, 29.25, 29.25, 28.4, 28.4, 35.9, 35.9 ])  # m
fc = np.concat( [6 * [f], 2* [fpr] ] )  # m flechas dos condutores 
yc = np.array(yc) - 2/3 * fc  # altura media dos condutores 
ri = 0.3365e-2  # raio interno dos condutores de fase (m)
rf = 0.9145e-2  # raio externo dos condutores de fase (m)
rdc = 0.1736e-3   # resistência DC dos condutores de fase (Ohm/m)
# rho_c = rdc * pi * (rf**2 - ri**2)  # resistividade efetiva dos condutores de fase (Ohm.m)
rpr = 0.0055245  # raio dos para-raios (m) considerando 3/8" EHS
rdc_pr =  2.8645e-3  # resistência DC dos para-raios (Ohm/m)
rho_s = 1e3          # restividade do solo em  (Ohm . m)

In [25]:
rpr

0.0055245

In [29]:
Zabc, Yabc = cZYlt(
    omega = 2j * pi * 60.0,
    x = xc[:6],
    y = yc[:6],
    sigmas = 1 / rho_s,
    rdc = rdc,
    rf = rf,
    rint = ri
)

/var/folders/xn/hgf3d_2j2xq6qfwjj8h4_n0r0000gn/T/ipykernel_61839/2046883021.py:92: RuntimeWarning: divide by zero encountered in divide
  Z_ext_off = 1j * omega * µ0 / (2 * pi) * 0.5 * log(numerator / denominator)
/var/folders/xn/hgf3d_2j2xq6qfwjj8h4_n0r0000gn/T/ipykernel_61839/2046883021.py:92: RuntimeWarning: invalid value encountered in multiply
  Z_ext_off = 1j * omega * µ0 / (2 * pi) * 0.5 * log(numerator / denominator)
/var/folders/xn/hgf3d_2j2xq6qfwjj8h4_n0r0000gn/T/ipykernel_61839/2046883021.py:114: RuntimeWarning: divide by zero encountered in divide
  P = 0.5 * log(num / den)


In [28]:
xc[:6]

array([-8.7286, -8.2714, -0.2286,  0.2286,  8.2714,  8.7286])

In [ ]:
Zabc, Yabc = cZYlt(
    omega = 2j * pi * 60.0,
    x = xc,
    y = yc,
    sigmas = 1 / rho_s,
    rdc = rdc,
    rf = rf,
    rint = ri,
    npr = 2,
    rdcpr = rdc_pr,
    rpr = rpr,
    nb = 2,
)

/var/folders/xn/hgf3d_2j2xq6qfwjj8h4_n0r0000gn/T/ipykernel_61839/2046883021.py:92: RuntimeWarning: divide by zero encountered in divide
  Z_ext_off = 1j * omega * µ0 / (2 * pi) * 0.5 * log(numerator / denominator)
/var/folders/xn/hgf3d_2j2xq6qfwjj8h4_n0r0000gn/T/ipykernel_61839/2046883021.py:92: RuntimeWarning: invalid value encountered in multiply
  Z_ext_off = 1j * omega * µ0 / (2 * pi) * 0.5 * log(numerator / denominator)
/var/folders/xn/hgf3d_2j2xq6qfwjj8h4_n0r0000gn/T/ipykernel_61839/2046883021.py:114: RuntimeWarning: divide by zero encountered in divide
  P = 0.5 * log(num / den)
